# GeoLift-S3 Lite + Teacher KD — TAR2000, 20 epochs

Ablation này giữ nguyên kiến trúc và bốn loss nền của S3 teacher-free. Thay đổi duy nhất là thêm metric KD từ `D_cm/C_cm` và fused-geometry KD từ `R_G/C_G`.

| Control | Teacher-KD run |
|---|---|
| Cùng `GeoLiftStudentS3Lite` | Có |
| Cùng MobileNetV4 ImageNet pretrained | Có |
| Cùng 1.600/400 split và seed 42 | Có |
| Cùng base loss S3 | Có |
| Cùng LR trajectory của baseline 30 epoch | Có; run dừng đánh giá sau epoch 19 |
| Khác biệt duy nhất | Metric + fused-geometry teacher supervision |

Objective: `L_total = L_S3-base + w_cm*L_metric-KD + w_G*L_SSI + w_ord*L_ordinal`.

| Epoch | w_cm | w_G | w_ord |
|---:|---:|---:|---:|
| 0 | 0.0667 | 0 | 0 |
| 1 | 0.1333 | 0 | 0 |
| 2 | 0.2000 | 0 | 0 |
| 3 | 0.2000 | 0.0100 | 0.0033 |
| 4 | 0.2000 | 0.0200 | 0.0067 |
| 5–19 | 0.2000 | 0.0300 | 0.0100 |

Input trên Drive:

```text
MyDrive/GeoLift_Data/teacher_subset_2000/
├── selected_2000_ids.json
├── kitti_trainval_2000.tar
├── metric_coarse_train_2000.tar
└── geometry_fused_train_2000.tar
```

## 0. Mount Drive và khai báo run

Chọn GPU runtime. Teacher và KITTI được đọc từ Drive nhưng extract sang SSD `/content`; checkpoint/log được backup về Drive sau mỗi epoch.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, re, io, json, shutil, tarfile, hashlib, subprocess
import numpy as np
import pandas as pd
import torch

drive.mount('/content/drive')
GITHUB_REPO = 'https://github.com/PhuocDang2104/GeoDistill_RT.git'
LOCAL_REPO = Path('/content/GeoDistill_RT')
TAR2000_ROOT = Path('/content/drive/MyDrive/GeoLift_Data/teacher_subset_2000')
FINAL_MANIFEST = TAR2000_ROOT / 'selected_2000_ids.json'
KITTI_TAR = TAR2000_ROOT / 'kitti_trainval_2000.tar'
METRIC_TAR = TAR2000_ROOT / 'metric_coarse_train_2000.tar'
GEOMETRY_TAR = TAR2000_ROOT / 'geometry_fused_train_2000.tar'
EXPECTED_KITTI_BYTES = 2_041_970_176

DATA_ROOT = Path('/content/geolift_s3_teacher_data')
KITTI_ROOT = DATA_ROOT / 'kitti_bundle'
TEACHER_ROOT = DATA_ROOT / 'teacher_outputs'
SPLIT_ROOT = DATA_ROOT / 'splits'
RUN_LOCAL = Path('/content/geolift_s3_teacher_kd_e20')
RUN_DRIVE = Path('/content/drive/MyDrive/GeoLift_RT_runs/v3_s3_lite_teacher_kd_train1600_val400_e20')
BASELINE_RUN_DRIVE = Path('/content/drive/MyDrive/GeoLift_RT_runs/v3_s3_lite_pretrained_train1600_val400')
EPOCHS, BATCH_SIZE, NUM_WORKERS = 20, 2, 2
FINAL_COUNT, TRAIN_COUNT, VAL_COUNT = 2000, 1600, 400
RESUME_TEACHER_RUN = True

if RUN_LOCAL.exists():
    shutil.rmtree(RUN_LOCAL)
for path in (DATA_ROOT, TEACHER_ROOT, SPLIT_ROOT, RUN_LOCAL, RUN_DRIVE):
    path.mkdir(parents=True, exist_ok=True)
for path in (FINAL_MANIFEST, KITTI_TAR, METRIC_TAR, GEOMETRY_TAR):
    assert path.is_file(), f'Thiếu input: {path}'
assert KITTI_TAR.stat().st_size == EXPECTED_KITTI_BYTES, KITTI_TAR.stat().st_size
assert torch.cuda.is_available(), 'Hãy chọn GPU runtime.'
payload_gib = sum(p.stat().st_size for p in (KITTI_TAR, METRIC_TAR, GEOMETRY_TAR)) / 1024**3
free_gib = shutil.disk_usage('/content').free / 1024**3
assert free_gib >= payload_gib + 12.0, (free_gib, payload_gib + 12.0)
print('GPU:', torch.cuda.get_device_name(0))
print('Run output:', RUN_DRIVE)
print(f'Local SSD {free_gib:.1f} GiB; required about {payload_gib + 12.0:.1f} GiB')

## 1. Clone source mới nhất vào Colab SSD

In [ ]:
if LOCAL_REPO.exists():
    shutil.rmtree(LOCAL_REPO)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main', GITHUB_REPO, str(LOCAL_REPO)], check=True)
os.chdir(LOCAL_REPO)
requirements = LOCAL_REPO / 'requirements.txt'
if requirements.is_file():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)], check=True)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Source commit:', commit)

## 2. Gate A — manifest và hai teacher TAR

Notebook bắt buộc hai TAR có đúng 2.000 canonical ID, đúng manifest và không trùng raw drive giữa train/validation.

In [ ]:
def canonical_id(member_name):
    stem = Path(member_name).stem
    old = re.match(r'^(?P<prefix>.+)_sync_image_(?P<camera>\d{2})_(?P<frame>\d{10})$', stem)
    if old:
        return f"{old.group('prefix')}_sync_image_{old.group('frame')}_image_{old.group('camera')}"
    canonical = re.match(r'^(?P<prefix>.+)_sync_image_(?P<frame>\d{10})_image_(?P<camera>\d{2})$', stem)
    if canonical:
        return stem
    raise ValueError(f'Không canonicalize được: {member_name}')

def raw_drive(sample_id):
    return sample_id.split('_sync_image_')[0]

def index_tar(path):
    result = {}
    with tarfile.open(path, 'r:*') as archive:
        for member in archive.getmembers():
            if not member.isfile() or not member.name.endswith('.npz'):
                continue
            sample_id = canonical_id(member.name)
            assert sample_id not in result, (path.name, sample_id)
            result[sample_id] = member.name
    print(path.name, 'NPZ:', len(result))
    return result

manifest = json.loads(FINAL_MANIFEST.read_text(encoding='utf-8'))
train_ids = list(manifest['train_ids'])
val_ids = list(manifest['val_ids'])
selected_ids = train_ids + val_ids
assert (len(train_ids), len(val_ids), len(set(selected_ids))) == (TRAIN_COUNT, VAL_COUNT, FINAL_COUNT)
assert not ({raw_drive(x) for x in train_ids} & {raw_drive(x) for x in val_ids})
metric_index = index_tar(METRIC_TAR)
geometry_index = index_tar(GEOMETRY_TAR)
assert set(metric_index) == set(geometry_index) == set(selected_ids)
print('Gate A passed: 2.000 matched teacher IDs; train/val raw-drive overlap = 0')

## 3. Extract teacher và KITTI bundle vào `/content`

In [ ]:
split_by_id = {x: 'train' for x in train_ids}
split_by_id.update({x: 'val' for x in val_ids})

def extract_teacher(tar_path, tar_index, role_root):
    for split in ('train', 'val'):
        (role_root / split).mkdir(parents=True, exist_ok=True)
    with tarfile.open(tar_path, 'r:*') as archive:
        members = {m.name: m for m in archive.getmembers()}
        ordered = sorted(((members[tar_index[x]].offset_data, x, members[tar_index[x]]) for x in selected_ids))
        for number, (_, sample_id, member) in enumerate(ordered, 1):
            source = archive.extractfile(member)
            assert source is not None
            target = role_root / split_by_id[sample_id] / f'{sample_id}.npz'
            with target.open('wb') as output:
                shutil.copyfileobj(source, output, length=8 * 1024 * 1024)
            if number % 200 == 0:
                print(role_root.name, number, '/', FINAL_COUNT)

if TEACHER_ROOT.exists():
    shutil.rmtree(TEACHER_ROOT)
extract_teacher(METRIC_TAR, metric_index, TEACHER_ROOT / 'metric_coarse')
extract_teacher(GEOMETRY_TAR, geometry_index, TEACHER_ROOT / 'geometry_fused')

def safe_extract_tar(source, destination):
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True)
    root = destination.resolve()
    with tarfile.open(source, 'r:*') as archive:
        members = archive.getmembers()
        for member in members:
            target = (destination / member.name).resolve()
            assert target == root or root in target.parents, member.name
        archive.extractall(destination, members=members)
safe_extract_tar(KITTI_TAR, KITTI_ROOT)

bundle_report = json.loads((KITTI_ROOT / 'kitti_bundle_report.json').read_text(encoding='utf-8'))
assert bundle_report['contract_ok'] is True
assert bundle_report['selected_count'] == FINAL_COUNT
bundle_manifest = json.loads((KITTI_ROOT / 'selected_2000_ids.json').read_text(encoding='utf-8'))
assert bundle_manifest['train_ids'] == manifest['train_ids']
assert bundle_manifest['val_ids'] == manifest['val_ids']
for name, expected in (('train_1600.txt', TRAIN_COUNT), ('val_400.txt', VAL_COUNT)):
    source = KITTI_ROOT / 'splits' / name
    lines = [x for x in source.read_text(encoding='utf-8').splitlines() if x.strip()]
    assert len(lines) == expected
    shutil.copy2(source, SPLIT_ROOT / name)
print('Teacher + KITTI extraction complete')

## 4. Gate B — coverage, schema và visualization

Coverage phải là 100% cho cả train và validation. Không cho phép Depth Anything fallback.

In [ ]:
coverage = {}
for role in ('metric_coarse', 'geometry_fused'):
    coverage[role] = {}
    for split, ids in (('train', train_ids), ('val', val_ids)):
        present = {p.stem for p in (TEACHER_ROOT / role / split).glob('*.npz')}
        coverage[role][split] = len(present & set(ids)) / len(ids)
        assert present == set(ids), (role, split, len(present))

for sample_id in train_ids[:12] + val_ids[:4]:
    split = split_by_id[sample_id]
    with np.load(TEACHER_ROOT / 'metric_coarse' / split / f'{sample_id}.npz', allow_pickle=False) as data:
        assert {'D_cm', 'C_cm'} <= set(data.files)
        D_cm, C_cm = np.asarray(data['D_cm']), np.asarray(data['C_cm'])
        assert D_cm.shape == C_cm.shape and np.isfinite(D_cm).all() and np.isfinite(C_cm).all()
        assert C_cm.min() >= 0 and C_cm.max() <= 1
    with np.load(TEACHER_ROOT / 'geometry_fused' / split / f'{sample_id}.npz', allow_pickle=False) as data:
        assert {'R_G', 'C_G'} <= set(data.files)
        R_G, C_G = np.asarray(data['R_G']), np.asarray(data['C_G'])
        assert R_G.shape == C_G.shape and np.isfinite(R_G).all() and np.isfinite(C_G).all()
        assert C_G.min() >= 0 and C_G.max() <= 1
teacher_report = {'contract_ok': True, 'coverage': coverage, 'inspected_schema_samples': 16,
                  'teacher_roles': ['metric_coarse', 'geometry_fused'], 'geometry_fallback': False}
(RUN_LOCAL / 'teacher_train_val_report.json').write_text(json.dumps(teacher_report,indent=2),encoding='utf-8')
print(json.dumps(teacher_report, indent=2))

import matplotlib.pyplot as plt
from src.dataset import KITTIDepthCompletionDataset
dataset = KITTIDepthCompletionDataset(
    data_root=KITTI_ROOT, split_root=SPLIT_ROOT, split_file='train_1600.txt', split_name='train',
    image_size=(352,1216), teacher_root=TEACHER_ROOT, load_teacher=True, load_geometry=True,
    geometry_fallback=False, load_mono=False, return_tensors=True,
)
sample = dataset[0]
assert {'D_cm','C_cm','R_G','C_G'} <= set(sample) and 'D_da_raw' not in sample
panels = [('RGB', sample['rgb'].permute(1,2,0), None), ('Sparse', sample['sparse'][0], 'turbo'),
          ('GT', sample['gt'][0], 'turbo'), ('D_cm', sample['D_cm'][0], 'turbo'),
          ('R_G', sample['R_G'][0], 'turbo'), ('C_G', sample['C_G'][0], 'viridis')]
fig, axes = plt.subplots(2,3,figsize=(20,8))
for ax,(title,image,cmap) in zip(axes.ravel(),panels):
    ax.imshow(image, cmap=cmap); ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()

## 5. Resolve config S3 Teacher-KD

Run bắt đầu từ cùng ImageNet pretrained initialization như baseline, không fine-tune từ checkpoint baseline. Resume chỉ dùng checkpoint của chính teacher-KD run.

In [ ]:
import yaml
paths_file = DATA_ROOT / 'paths_s3_teacher_kd.yaml'
paths = {
    'data_root': str(KITTI_ROOT), 'split_root': str(SPLIT_ROOT),
    'train_split': 'train_1600.txt', 'val_split': 'val_400.txt', 'test_split': 'val_400.txt',
    'teacher_root': str(TEACHER_ROOT), 'student_root': str(RUN_LOCAL),
}
paths_file.write_text(yaml.safe_dump(paths, sort_keys=False), encoding='utf-8')
base_config = LOCAL_REPO / 'configs' / 'geolift_s3_lite_teacher_kd_tar2000.yaml'
assert base_config.is_file(), 'Hãy push source mới có config teacher-KD lên GitHub trước.'
cfg = yaml.safe_load(base_config.read_text(encoding='utf-8'))
cfg['paths_file'] = str(paths_file)
cfg['train']['epochs'] = EPOCHS
cfg['train']['batch_size'] = BATCH_SIZE
cfg['data']['num_workers'] = NUM_WORKERS
cfg['outputs']['backup_root'] = str(RUN_DRIVE)
cfg['train']['resume'] = None
last_drive = RUN_DRIVE / 'checkpoints' / 'last.pth'
if RESUME_TEACHER_RUN and last_drive.is_file():
    previous_config_file = RUN_DRIVE / 'resolved_config.yaml'
    if previous_config_file.is_file():
        previous_cfg = yaml.safe_load(previous_config_file.read_text(encoding='utf-8'))
        assert previous_cfg['model'] == cfg['model'], 'Resume model config mismatch'
        assert previous_cfg['loss'] == cfg['loss'], 'Resume loss config mismatch'
        assert previous_cfg['schedule'] == cfg['schedule'], 'Resume schedule config mismatch'
        for key in ('epochs','batch_size','lr','scheduler','scheduler_total_epochs'):
            assert previous_cfg['train'][key] == cfg['train'][key], ('Resume train config mismatch',key)
    local_last = RUN_LOCAL / 'checkpoints' / 'last.pth'
    local_last.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(last_drive, local_last)
    best_drive = RUN_DRIVE / 'checkpoints' / 'best.pth'
    if best_drive.is_file():
        shutil.copy2(best_drive, local_last.parent / 'best.pth')
    local_logs = RUN_LOCAL / 'logs'
    local_logs.mkdir(parents=True, exist_ok=True)
    for name in ('train_log.csv','train_log.jsonl','train_student.log'):
        source = RUN_DRIVE / 'logs' / name
        if source.is_file():
            shutil.copy2(source, local_logs / name)
    cfg['train']['resume'] = str(local_last)
resolved_cfg = DATA_ROOT / 'geolift_s3_teacher_kd_e20.yaml'
resolved_cfg.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
run_manifest = {
    'architecture': 'GeoLift-S3-Lite', 'experiment': 'teacher_kd_only', 'epochs': EPOCHS,
    'teacher_roles': ['metric_coarse:D_cm/C_cm', 'geometry_fused:R_G/C_G'],
    'metric_kd_scope': 'non_KITTI_GT_pixels_only', 'geometry_fallback': False,
    'base_objective_unchanged': True, 'baseline_run': str(BASELINE_RUN_DRIVE),
    'scheduler_total_epochs': cfg['train']['scheduler_total_epochs'],
    'teacher_tar_bytes': {'metric': METRIC_TAR.stat().st_size, 'geometry': GEOMETRY_TAR.stat().st_size},
    'train': TRAIN_COUNT, 'val': VAL_COUNT, 'source_commit': commit,
    'config_sha256': hashlib.sha256(resolved_cfg.read_bytes()).hexdigest(),
}
artifacts = [(resolved_cfg,'resolved_config.yaml'), (paths_file,'resolved_paths.yaml'),
             (FINAL_MANIFEST,'selected_2000_ids.json')]
for source,name in artifacts:
    shutil.copy2(source, RUN_LOCAL / name)
(RUN_LOCAL / 'run_manifest.json').write_text(json.dumps(run_manifest,indent=2),encoding='utf-8')
subprocess.run(['rsync','-a',f'{RUN_LOCAL}/',f'{RUN_DRIVE}/'],check=True)
print(resolved_cfg.read_text(encoding='utf-8'))

## 6. Contract tests và real-batch smoke

Cell này xác nhận model vẫn có đúng 369.209 parameters, batch chứa đủ hai teacher role và teacher weights bật đúng lịch.

In [ ]:
subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-v'],cwd=LOCAL_REPO,check=True)
from src.utils import load_project_config
from src.train_student import make_loader, preflight_teacher_coverage
from src.model_factory import build_student
from src.losses import geort_loss
import logging
loaded_cfg, loaded_paths = load_project_config(resolved_cfg)
loader = make_loader(loaded_cfg, loaded_paths, 'train', training=True)
preflight_teacher_coverage(loaded_cfg, loader.dataset, logging.getLogger('preflight'))
batch = next(iter(loader))
assert {'D_cm','C_cm','R_G','C_G'} <= set(batch) and 'D_da_raw' not in batch
model = build_student(loaded_cfg).cuda().train()
assert sum(p.numel() for p in model.parameters()) == 369_209
gpu_batch = {k:(v[:1].cuda() if torch.is_tensor(v) else v) for k,v in batch.items()}
with torch.autocast('cuda',dtype=torch.float16):
    pred = model(*(gpu_batch[k] for k in ('rgb','sparse','mask','ray','uv','K')))
    loss0, items0 = geort_loss(pred,gpu_batch,loaded_cfg['loss'],loaded_cfg['schedule'],0,loaded_cfg['mono_ssi'])
    loss3, items3 = geort_loss(pred,gpu_batch,loaded_cfg['loss'],loaded_cfg['schedule'],3,loaded_cfg['mono_ssi'])
assert torch.isfinite(loss0) and torch.isfinite(loss3)
assert items0['w_teacher_metric'] > 0 and items0['w_teacher_geometry'] == 0
assert items3['w_teacher_geometry'] > 0 and items3['w_teacher_ordinal'] > 0
assert items0['teacher_metric_non_gt_ratio'] > 0
loss3.backward()
assert all(p.grad is None or torch.isfinite(p.grad).all() for p in model.parameters())
print('Smoke passed:', {k:items3[k] for k in items3 if k.startswith('w_teacher')})
del model,pred,batch,gpu_batch,loader
torch.cuda.empty_cache()

## 7. Train 20 epochs

Không nạp checkpoint baseline. Nếu `RESUME_TEACHER_RUN=True`, notebook chỉ resume checkpoint của chính run teacher-KD.

In [ ]:
train_command = [sys.executable,'-m','src.train_student','--config',str(resolved_cfg)]
try:
    subprocess.run(train_command,cwd=LOCAL_REPO,check=True)
finally:
    subprocess.run(['rsync','-a',f'{RUN_LOCAL}/',f'{RUN_DRIVE}/'],check=True)
print('Training complete:', RUN_DRIVE)

## 8. Final validation inference và FP16 profile

In [ ]:
best = RUN_LOCAL / 'checkpoints' / 'best.pth'
assert best.is_file(), best
subprocess.run([sys.executable,'-m','src.infer_student','--config',str(resolved_cfg),'--checkpoint',str(best),'--split','val'],cwd=LOCAL_REPO,check=True)
profile_out = RUN_LOCAL / 'logs' / 'geolift_s3_component_profile.json'
subprocess.run([sys.executable,'scripts/profile_geolift_s3.py','--config',str(resolved_cfg),'--warmup','20','--runs','100','--output',str(profile_out)],cwd=LOCAL_REPO,check=True)
subprocess.run(['rsync','-a',f'{RUN_LOCAL}/',f'{RUN_DRIVE}/'],check=True)
print((RUN_LOCAL / 'logs' / 'infer_val_metrics_global.json').read_text(encoding='utf-8'))
print(profile_out.read_text(encoding='utf-8'))

## 9. So sánh công bằng với 20 epoch đầu của baseline

Cả hai phía đều chỉ xét epoch `0…19`; best checkpoint được chọn bằng global validation RMSE.

In [ ]:
import matplotlib.pyplot as plt
teacher_log = RUN_LOCAL / 'logs' / 'train_log.csv'
baseline_log = BASELINE_RUN_DRIVE / 'logs' / 'train_log.csv'
baseline_config_file = BASELINE_RUN_DRIVE / 'resolved_config.yaml'
assert teacher_log.is_file(), teacher_log
teacher_df = pd.read_csv(teacher_log)
teacher_df = teacher_df[teacher_df.epoch < EPOCHS].copy()
teacher_best = teacher_df.loc[teacher_df.val_rmse.idxmin()]
if baseline_log.is_file():
    if baseline_config_file.is_file():
        baseline_cfg = yaml.safe_load(baseline_config_file.read_text(encoding='utf-8'))
        assert baseline_cfg['model']['architecture'] == cfg['model']['architecture'] == 'geolift_s3_lite'
        assert baseline_cfg['model']['encoder'] == cfg['model']['encoder']
        assert baseline_cfg['data']['image_size'] == cfg['data']['image_size']
        for key in ('lambda_metric','lambda_log','lambda_sparse','lambda_edge','multiscale_weights'):
            assert baseline_cfg['loss'][key] == cfg['loss'][key], ('base loss mismatch',key)
        assert int(baseline_cfg['train']['epochs']) == int(cfg['train']['scheduler_total_epochs']) == 30
        print('Baseline architecture/base-loss/LR-horizon gate passed')
    baseline_df = pd.read_csv(baseline_log)
    baseline_df = baseline_df[baseline_df.epoch < EPOCHS].copy()
    assert len(baseline_df) >= EPOCHS, f'Baseline chỉ có {len(baseline_df)} epoch trong 0..19'
    baseline_best = baseline_df.loc[baseline_df.val_rmse.idxmin()]
    metrics = ['val_rmse','val_mae','val_irmse','val_abs_rel','val_delta1',
               'val_rmse_0_20','val_rmse_20_40','val_rmse_40_60','val_rmse_60_80','val_rmse_80_120',
               'val_rmse_edge','val_rmse_nonedge']
    rows = []
    for metric in metrics:
        b, t = float(baseline_best[metric]), float(teacher_best[metric])
        lower_is_better = metric != 'val_delta1'
        gain = (b-t)/max(abs(b),1e-12)*100 if lower_is_better else (t-b)/max(abs(b),1e-12)*100
        rows.append({'metric':metric,'baseline_e0_19_best':b,'teacher_kd_e0_19_best':t,'gain_pct':gain})
    comparison = pd.DataFrame(rows)
    comparison.to_csv(RUN_LOCAL / 'teacher_kd_vs_teacher_free_e20.csv',index=False)
    display(comparison)
    fig,axes=plt.subplots(1,2,figsize=(14,5))
    axes[0].plot(baseline_df.epoch,baseline_df.val_rmse,label='S3 teacher-free')
    axes[0].plot(teacher_df.epoch,teacher_df.val_rmse,label='S3 + teacher KD')
    axes[0].set(xlabel='Epoch',ylabel='Validation RMSE (m)',title='Same architecture, first 20 epochs'); axes[0].grid(); axes[0].legend()
    range_cols=['val_rmse_0_20','val_rmse_20_40','val_rmse_40_60','val_rmse_60_80']
    x=np.arange(len(range_cols)); axes[1].bar(x-.2,[baseline_best[c] for c in range_cols],.4,label='teacher-free')
    axes[1].bar(x+.2,[teacher_best[c] for c in range_cols],.4,label='teacher KD')
    axes[1].set_xticks(x,['0-20','20-40','40-60','60-80']); axes[1].set(ylabel='RMSE (m)',xlabel='GT range (m)',title='Best-by-global-RMSE checkpoint'); axes[1].legend()
    fig.tight_layout(); fig.savefig(RUN_LOCAL / 'teacher_kd_vs_teacher_free_e20.png',dpi=180,bbox_inches='tight'); plt.show()
    print('Baseline best epoch:',int(baseline_best.epoch),'RMSE:',float(baseline_best.val_rmse))
else:
    print('Không tìm thấy baseline log:', baseline_log)
print('Teacher-KD best epoch:',int(teacher_best.epoch),'RMSE:',float(teacher_best.val_rmse))
loss_budget = pd.DataFrame({
    'term':['S3 base','metric teacher KD','geometry SSI','geometry ordinal'],
    'weighted_value':[
        float(teacher_best.train_loss) - float(teacher_best.train_w_teacher_metric)*float(teacher_best.train_L_teacher_metric_kd) - float(teacher_best.train_w_teacher_geometry)*float(teacher_best.train_L_teacher_geometry_ssi) - float(teacher_best.train_w_teacher_ordinal)*float(teacher_best.train_L_teacher_geometry_ord),
        float(teacher_best.train_w_teacher_metric)*float(teacher_best.train_L_teacher_metric_kd),
        float(teacher_best.train_w_teacher_geometry)*float(teacher_best.train_L_teacher_geometry_ssi),
        float(teacher_best.train_w_teacher_ordinal)*float(teacher_best.train_L_teacher_geometry_ord),
    ]})
loss_budget.to_csv(RUN_LOCAL / 'teacher_kd_loss_budget.csv',index=False); display(loss_budget)
subprocess.run(['rsync','-a',f'{RUN_LOCAL}/',f'{RUN_DRIVE}/'],check=True)

## Drive output

```text
MyDrive/GeoLift_RT_runs/v3_s3_lite_teacher_kd_train1600_val400_e20/
├── checkpoints/{best.pth,last.pth,epoch_*.pth}
├── logs/{train_log.csv,train_log.jsonl,train_student.log}
├── logs/{infer_val_metrics_global.json,geolift_s3_component_profile.json}
├── teacher_kd_vs_teacher_free_e20.csv
├── teacher_kd_vs_teacher_free_e20.png
├── teacher_kd_loss_budget.csv
├── teacher_train_val_report.json
├── resolved_config.yaml
├── resolved_paths.yaml
└── run_manifest.json
```

Run này không tạo anonymous-test ZIP vì tập test 1.000 không có public GT và không giúp đo teacher ablation.